# Relative Effects and MDE for Ratio Metrics

This notebook demonstrates the difference between Absolute and Relative inference and Minimum Detectable Effect (MDE) calculations for ratio metrics (e.g., Conversion Rate, Units per Order) using the Cluster Experiments package.

When dealing with relative effects on ratio metrics, standard OLS delta methods are insufficient. The variance of the estimated relative effect must account for the uncertainty in *both* the absolute treatment effect (numerator) and the control group mean (denominator). 

In [1]:
import numpy as np
import pandas as pd

from cluster_experiments.experiment_analysis import DeltaMethodAnalysis
from cluster_experiments.power_analysis import NormalPowerAnalysis
from cluster_experiments.random_splitter import ClusteredSplitter

import warnings
warnings.filterwarnings('ignore')

## 1. Simulating Data
We'll simulate a scenario where assigning treatment to users (`cluster_col`) affects their `target` (e.g., clicks) scaled by some `scale` (e.g., impressions), measuring the ratio metric `target / scale`.

In [2]:
# Simulate some user-level data
np.random.seed(42)
n_users = 2000
n_rows = 10000

df = pd.DataFrame({
    "user": np.random.choice([f"User {i}" for i in range(n_users)], size=n_rows),
    "scale": np.random.poisson(lam=10, size=n_rows).clip(1), # Impressions
})

# Add noise and baseline behavior
user_effects = {f"User {i}": np.random.normal(0, 0.1) for i in range(n_users)}
df["user_effect"] = df["user"].map(user_effects)
df["baseline_prob"] = 0.05 + df["user_effect"]
df["baseline_prob"] = df["baseline_prob"].clip(0.01, 0.99)

# Apply treatment 
splitter = ClusteredSplitter(cluster_cols=["user"])
df = splitter.assign_treatment_df(df)
df["is_treatment"] = (df["treatment"] == "B").astype(int)

# Treatment effect: +15% relative lift
TRUE_RELATIVE_LIFT = 0.15
df["prob"] = df["baseline_prob"] * (1 + TRUE_RELATIVE_LIFT * df["is_treatment"])

# Generate target events (e.g., Clicks)
df["target"] = np.random.binomial(df["scale"], df["prob"])

df.head()

,user,scale,user_effect,baseline_prob,treatment,is_treatment,prob,target
0,User 1126,10,-0.037478,0.012522,A,0,0.012522,1
1,User 1459,13,0.030119,0.080119,A,0,0.080119,0
2,User 860,15,0.012006,0.062006,A,0,0.062006,1
3,User 1294,10,-0.088254,0.010000,A,0,0.010000,0
4,User 1130,8,0.076286,0.126286,B,1,0.145229,1


## 2. Naive Relative vs Proper Delta Method

Let's look at the Point Estimate and the Standard Error (SE).

In [3]:
analyser_abs = DeltaMethodAnalysis(
    cluster_cols=["user"], scale_col="scale", target_col="target", relative_effect=False
)
analyser_rel = DeltaMethodAnalysis(
    cluster_cols=["user"], scale_col="scale", target_col="target", relative_effect=True
)

# 1. Absolute Inference
abs_ate = analyser_abs.get_point_estimate(df)
abs_se = analyser_abs.get_standard_error(df)

# 2. Proper Relative Inference (Outer Delta Method)
rel_ate = analyser_rel.get_point_estimate(df)
rel_se = analyser_rel.get_standard_error(df)

# 3. Naive Relative Inference (Divide Absolute values by Control Mean)
# Calculate Control Mean:
ctrl_df = df[df["treatment"] == "A"]
ctrl_mean = ctrl_df["target"].sum() / ctrl_df["scale"].sum()

naive_rel_ate = abs_ate / ctrl_mean
naive_rel_se = abs_se / ctrl_mean

print(f"Control Mean: {ctrl_mean:.4f}")
print("-" * 40)
print(f"Absolute ATE: {abs_ate:.4f}")
print(f"Absolute SE:  {abs_se:.4f}")
print("-" * 40)
print(f"Naive Relative ATE:  {naive_rel_ate:.4f}  (Absolute ATE / Control Mean)")
print(f"Proper Relative ATE: {rel_ate:.4f}  (Matches Naive)")
print("-" * 40)
print(f"Naive Relative SE:  {naive_rel_se:.4f}  (Absolute SE / Control Mean) -> UNDERESTIMATES VARIANCE")
print(f"Proper Relative SE: {rel_se:.4f}  (Outer Delta Method)")

Control Mean: 0.0796
----------------------------------------
Absolute ATE: 0.0015
Absolute SE:  0.0041
----------------------------------------
Naive Relative ATE:  0.0186  (Absolute ATE / Control Mean)
Proper Relative ATE: 0.0186  (Matches Naive)
----------------------------------------
Naive Relative SE:  0.0520  (Absolute SE / Control Mean) -> UNDERESTIMATES VARIANCE
Proper Relative SE: 0.0525  (Outer Delta Method)


Notice that the point estimates perfectly match (the outer delta method preserves the estimator's mean), but the **Proper Relative SE is strictly larger** than the naive calculation.

The naive calculation assumes the denominator (the control metric) is a fixed constant, effectively ignoring the natural statistical noise present in measuring the control behavior. The Delta method incorporates this variance into the SE calculation, resulting in more truthful, slightly wider confidence intervals.

## 3. Absolute MDE vs Relative MDE
Because the relative variance scales quadratically with respect to the effect size itself, solving for the Minimum Detectable Effect becomes a polynomial root-finding exercise (Double Delta Method).

In [4]:
pw_abs = NormalPowerAnalysis(
    splitter=splitter, analysis=analyser_abs, n_simulations=50
)
pw_rel = NormalPowerAnalysis(
    splitter=splitter, analysis=analyser_rel, n_simulations=50
)

# The Absolute MDE tells us the raw lift in the ratio required to reach 80% power
mde_abs = pw_abs.mde(df, power=0.8)

# The Relative MDE solves the actual quadratic polynomial to find the % lift required to reach 80% power
mde_rel = pw_rel.mde(df, power=0.8)

naive_mde_rel = mde_abs / ctrl_mean

print(f"Absolute MDE: {mde_abs:.4f}")
print("-" * 40)
print(f"Naive Relative MDE:  {naive_mde_rel:.4f}  (Absolute MDE / Control Mean) -> OVEROPTIMISTIC")
print(f"Proper Relative MDE: {mde_rel:.4f}  (Double Delta Method)")


Absolute MDE: 0.0116
----------------------------------------
Naive Relative MDE:  0.1458  (Absolute MDE / Control Mean) -> OVEROPTIMISTIC
Proper Relative MDE: 0.1474  (Double Delta Method)


The required relative effect to achieve 80% power is slightly larger than simply scaling the absolute MDE, because detecting large relative effects is penalized by the expanding variance. 